# Windowing and Dataset Splitting for Sequence Modeling

This notebook converts engineered tabular data into fixed-length temporal windows suitable for sequence models (for example LSTM/CNN).

## Objectives
- Define the window length and overlap strategy.
- Transform sensor time series into supervised learning samples.
- Keep labels aligned with the end of each window.
- Split windows into train/validation/test sets with stratification.
- Save all arrays for downstream model training.

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

DATA_DIR = "../data"

## Load Engineered Dataset
The next step loads data_engineered.csv from the feature-engineering stage and parses timestamps for temporal ordering checks.

This ensures we are windowing the fully prepared feature set rather than raw signals.

In [10]:
df = pd.read_csv(f"{DATA_DIR}/data_engineered.csv")
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

print(f"Shape  : {df.shape}")
print(f"Leaks  : {df['Leak Status'].sum()}")
df.head()


Shape  : (990, 9)
Leaks  : 19


,Timestamp,Sensor_ID,Pressure (bar),Flow Rate (L/s),Leak Status,pressure_delta,flow_zscore,pf_ratio,sensor_enc
0,2024-01-01 01:10:00,S001,2.595338,96.647348,0,0.000000,-0.560187,0.026854,0
1,2024-01-01 02:20:00,S001,2.726076,126.229817,0,0.130739,0.089423,0.021596,0
2,2024-01-01 02:30:00,S001,3.096358,127.662703,0,0.370282,0.120888,0.024254,0
3,2024-01-01 05:15:00,S001,3.477227,81.002654,0,0.380869,-0.903734,0.042927,0
4,2024-01-01 05:45:00,S001,3.257879,173.968620,0,-0.219349,1.137734,0.018727,0


## Configure Window Parameters
Here we define the temporal context length and stride.

With WINDOW_SIZE = 20 and STEP_SIZE = 5, each sample covers 100 minutes and consecutive windows overlap by 75%.

In [11]:
WINDOW_SIZE = 20   # 20 timesteps = 100 minutes of sensor history
STEP_SIZE   = 5    # step of 5 = 75% overlap between windows

FEATURES = ['Pressure (bar)', 'Flow Rate (L/s)',
            'pressure_delta', 'flow_zscore',
            'pf_ratio', 'sensor_enc']

print(f"Window size : {WINDOW_SIZE} timesteps ({WINDOW_SIZE * 5} minutes)")
print(f"Step size   : {STEP_SIZE} timesteps ({STEP_SIZE * 5} minutes)")
print(f"Features    : {len(FEATURES)} → {FEATURES}")


Window size : 20 timesteps (100 minutes)
Step size   : 5 timesteps (25 minutes)
Features    : 6 → ['Pressure (bar)', 'Flow Rate (L/s)', 'pressure_delta', 'flow_zscore', 'pf_ratio', 'sensor_enc']


## Build Sliding Windows
This block iterates sensor-by-sensor, preserving time order, and creates overlapping windows over the selected features.

Labeling rule: each window gets the label of its last timestep, so prediction targets the current state at window end.

In [12]:
X_list, y_list, meta_list = [], [], []

for sensor_id, group in df.groupby('Sensor_ID'):
    group = group.sort_values('Timestamp').reset_index(drop=True)
    vals   = group[FEATURES].values
    labels = group['Leak Status'].values

    for start in range(0, len(group) - WINDOW_SIZE + 1, STEP_SIZE):
        end = start + WINDOW_SIZE
        X_list.append(vals[start:end])
        # ANY leak inside the window = positive label
        y_list.append(int(labels[start:end].any()))
        meta_list.append({
            'sensor_id' : sensor_id,
            'start_idx' : start,
            'end_idx'   : end - 1
        })

X = np.array(X_list)
y = np.array(y_list)

print(f"X shape        : {X.shape}")
print(f"y shape        : {y.shape}")
print(f"Normal windows : {(y==0).sum()}")
print(f"Leak windows   : {(y==1).sum()}")
print(f"Leak ratio     : {y.mean()*100:.2f}%")


X shape        : (164, 20, 6)
y shape        : (164,)
Normal windows : 112
Leak windows   : 52
Leak ratio     : 31.71%


## Inspect Example Windows
To sanity-check feature behavior, we print one normal and one leak window.

This helps verify that generated sequences and feature scales look reasonable before training.

In [13]:
# Print one Normal and one Leak window side by side
normal_idx = np.where(y==0)[0][0]
leak_idx   = np.where(y==1)[0][0]

print("=== NORMAL WINDOW ===")
print(pd.DataFrame(X[normal_idx], columns=FEATURES).round(4))

print("\n=== LEAK WINDOW ===")
print(pd.DataFrame(X[leak_idx], columns=FEATURES).round(4))


=== NORMAL WINDOW ===
    Pressure (bar)  Flow Rate (L/s)  pressure_delta  flow_zscore  pf_ratio  \
0           2.5953          96.6473          0.0000      -0.5602    0.0269   
1           2.7261         126.2298          0.1307       0.0894    0.0216   
2           3.0964         127.6627          0.3703       0.1209    0.0243   
3           3.4772          81.0027          0.3809      -0.9037    0.0429   
4           3.2579         173.9686         -0.2193       1.1377    0.0187   
5           3.8205          85.5028          0.5627      -0.8049    0.0447   
6           2.9196         111.6810         -0.9009      -0.2301    0.0261   
7           3.7575         120.3040          0.8378      -0.0407    0.0312   
8           3.3279         134.1407         -0.4296       0.2631    0.0248   
9           3.9243          72.0610          0.5964      -1.1001    0.0545   
10          2.9153         170.9302         -1.0090       1.0710    0.0171   
11          3.4159          93.2946       

## Stratified Train/Validation/Test Split
Windows are split with stratification so leak prevalence stays similar across subsets.

This reduces distribution mismatch risk and makes evaluation more reliable.

In [14]:
X_train_list, y_train_list = [], []
X_val_list,   y_val_list   = [], []
X_test_list,  y_test_list  = [], []

for sensor_id, group in df.groupby('Sensor_ID'):
    group  = group.sort_values('Timestamp').reset_index(drop=True)
    vals   = group[FEATURES].values
    labels = group['Leak Status'].values

    # Build all windows for this sensor
    sensor_X, sensor_y = [], []
    for start in range(0, len(group) - WINDOW_SIZE + 1, STEP_SIZE):
        end = start + WINDOW_SIZE
        sensor_X.append(vals[start:end])
        sensor_y.append(int(labels[start:end].any()))

    n         = len(sensor_X)
    train_end = int(0.70 * n)
    val_end   = int(0.85 * n)

    # Temporal split: early → train, middle → val, late → test
    X_train_list.extend(sensor_X[:train_end])
    y_train_list.extend(sensor_y[:train_end])

    X_val_list.extend(sensor_X[train_end:val_end])
    y_val_list.extend(sensor_y[train_end:val_end])

    X_test_list.extend(sensor_X[val_end:])
    y_test_list.extend(sensor_y[val_end:])

X_train = np.array(X_train_list)
X_val   = np.array(X_val_list)
X_test  = np.array(X_test_list)
y_train = np.array(y_train_list)
y_val   = np.array(y_val_list)
y_test  = np.array(y_test_list)

print("=== SPLIT SUMMARY ===")
print(f"Train : {X_train.shape[0]} windows  | Leaks = {y_train.sum()}")
print(f"Val   : {X_val.shape[0]} windows  | Leaks = {y_val.sum()}")
print(f"Test  : {X_test.shape[0]} windows  | Leaks = {y_test.sum()}")


=== SPLIT SUMMARY ===
Train : 111 windows  | Leaks = 37
Val   : 23 windows  | Leaks = 7
Test  : 30 windows  | Leaks = 8


## Verify Class Balance Across Splits
This check compares leak ratios in train, validation, and test sets against the full dataset.

Close percentages indicate stratification worked as expected.

In [15]:
print("=== LEAK RATIO PER SPLIT ===")
print(f"Overall : {y.mean()*100:.2f}%")
print(f"Train   : {y_train.mean()*100:.2f}%")
print(f"Val     : {y_val.mean()*100:.2f}%")
print(f"Test    : {y_test.mean()*100:.2f}%")


=== LEAK RATIO PER SPLIT ===
Overall : 31.71%
Train   : 33.33%
Val     : 30.43%
Test    : 26.67%


## Save Windowed Arrays
The following step writes all split arrays to disk as .npy files for efficient loading during model training.

Saved artifacts include X_train/X_val/X_test and y_train/y_val/y_test.

In [16]:
np.save(f"{DATA_DIR}/X_train.npy", X_train)
np.save(f"{DATA_DIR}/X_val.npy",   X_val)
np.save(f"{DATA_DIR}/X_test.npy",  X_test)
np.save(f"{DATA_DIR}/y_train.npy", y_train)
np.save(f"{DATA_DIR}/y_val.npy",   y_val)
np.save(f"{DATA_DIR}/y_test.npy",  y_test)

print("All splits saved to data/")
print(f"   X_train.npy → {X_train.shape}")
print(f"   X_val.npy   → {X_val.shape}")
print(f"   X_test.npy  → {X_test.shape}")
print(f"   y_train.npy → {y_train.shape}")
print(f"   y_val.npy   → {y_val.shape}")
print(f"   y_test.npy  → {y_test.shape}")


All splits saved to data/
   X_train.npy → (111, 20, 6)
   X_val.npy   → (23, 20, 6)
   X_test.npy  → (30, 20, 6)
   y_train.npy → (111,)
   y_val.npy   → (23,)
   y_test.npy  → (30,)


## Windowing Takeaways
- Each sample now contains fixed-length multivariate temporal context.
- Labels are aligned to the final timestep of each window.
- Stratified splits preserve leak proportions for fair evaluation.
- Saved NumPy arrays are ready for the modeling notebook.